In [1]:
import pandas as pd
import numpy as np
from pymc import find_constrained_prior, LogNormal
import json

We want to choose priors such that 95% of the probability lies between the upper and lower bounds. In particular we will choose lognormal priors are they are naturally constrained to the positive reals. PyMC's `find_constrained_prior` function allows us to specify a distribution, bounds, and percent of mass between the bounds, and then returns the parameters of the prior that approximately place the probability mass between the bounds.

The lognormal distriubtion has a pdf
$$
    f(x| \mu, \tau) = \frac{1}{x}\sqrt{\frac{\tau}{2\pi}}\exp \left\{-\frac{\tau}{2}(\ln(x) - \mu)^2 \right\}
$$
with mean $\exp\{\mu + \frac{1}{2\tau}\}$ and variance $(\exp\{ \frac{1}{\tau}\} - 1) \times \exp\{ 2\mu + \frac{1}{\tau}\}$.

PyMC allows us to specify $\mu$ and $\tau$ or $\sigma$, but not both $\tau$ and $\sigma$.

First, we want to load the bounds, and nominal value parameters to construct the priors.

In [2]:
bounds_MA = pd.read_csv('../global_sensitivity_analysis/param_bounds_MA.csv')
bounds_MM = pd.read_csv('../global_sensitivity_analysis/param_bounds_MM.csv')
bounds_newmech_MA = pd.read_csv('../global_sensitivity_analysis/param_bounds_newmech_MA.csv')
# get list of names
names_MA = bounds_MA['parameter'].to_list()
names_MM = bounds_MM['parameter'].to_list()
names_newmech_MA = bounds_newmech_MA['parameter'].to_list()
# list of bounds
bounds_MA = [[lb, ub] for lb, ub in zip(bounds_MA['lb'].to_list(), bounds_MA['ub'].to_list())]
bounds_MM = [[lb, ub] for lb, ub in zip(bounds_MM['lb'].to_list(), bounds_MM['ub'].to_list())]
bounds_newmech_MA = [[lb, ub] for lb, ub in zip(bounds_newmech_MA['lb'].to_list(), bounds_newmech_MA['ub'].to_list())]

In [3]:
tau_init = 1.0
mu_logN = lambda mean, tau: np.log(mean) + (1/(2*tau)) # function to make a starting guess of mu st mean is the middle of the bounds
sigma = lambda upper, lower: np.sqrt((upper-lower)/(12))
# MA
MA_prior_params = {}
for name, bound in zip(names_MA, bounds_MA):
    init_guess = {'mu': mu_logN((bound[1]-bound[0])/2, tau_init), 'tau': tau_init}
    try:
        MA_prior_params[name] = find_constrained_prior(LogNormal, bound[0], bound[1], init_guess=init_guess,
                                                       mass=0.5)
    except:
        print('Optimization failed for {}'.format(name))
        MA_prior_params[name] = np.nan

with open('MA_prior_params.json', 'w') as file:
    json.dump(MA_prior_params, file)

# MM
MM_prior_params = {}
for name, bound in zip(names_MM, bounds_MM):
    init_guess = {'mu': mu_logN((bound[1]-bound[0])/2, tau_init), 'tau': tau_init}
    try:
        MM_prior_params[name] = find_constrained_prior(LogNormal, bound[0], bound[1], init_guess=init_guess,
                                                       mass=0.5)
    except:
        print('Optimization failed for {}'.format(name))
        MM_prior_params[name] = np.nan

with open('MM_prior_params.json', 'w') as file:
    json.dump(MM_prior_params, file)

# newmech_MA
newmech_MA_prior_params = {}
for name, bound in zip(names_newmech_MA, bounds_newmech_MA):
    init_guess = {'mu': mu_logN((bound[1]-bound[0])/2, tau_init), 'tau': tau_init}
    try:
        newmech_MA_prior_params[name] = find_constrained_prior(LogNormal, bound[0], bound[1], init_guess=init_guess,
                                                               mass=0.5)
    except:
        print('Optimization failed for {}'.format(name))
        newmech_MA_prior_params[name] = np.nan

with open('newmech_MA_prior_params.json', 'w') as file:
    json.dump(newmech_MA_prior_params, file)